# A/B Testing Player Retention

The dataset contains the result of an A/B test with a popular mobile game, Cookie Cats.

Users were randomly assigned to a game gate - a progression barrier that forces players to wait or pay to continue playing - at level 30 or 40. Outcomes include 1-day retention, 7-day retention, and game rounds.
 
The business question is fairly simple: **Should the game move its progression gate from level 30 to level 40**?

Link to the data source: https://www.datacamp.com/projects/184

#### 1. Library and Data import

In [69]:
import kagglehub
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

path = kagglehub.dataset_download("yufengsui/mobile-games-ab-testing")
filename = path + "/cookie_cats.csv"
df = pd.read_csv(filename)

#### 2. EDA

* The dataset contains 90,189 entries, each corresponding to a unique player and no null values were found

* Treatment and control groups were similar in size, roughly 50-50 split, suggesting no obvious issue with treatment allocation

| Variable | Description |
|---|---|
| `userid` | A unique number that identifies each player. |
| `version` | Whether the player was assigned to the control group (`gate_30`, with the gate at level 30) or the treatment group (`gate_40`, with the gate moved to level 40). |
| `sum_gamerounds` | The number of game rounds played by the player during the first 14 days after installation. |
| `retention_1` | Whether the player returned and played 1 day after installing the game. |
| `retention_7` | Whether the player returned and played 7 days after installing the game. |

In [62]:
print("Number of unique users:", df["userid"].nunique())

summary = pd.DataFrame(
{
    "count": df["version"].value_counts(),
    "percentage": df["version"].value_counts(normalize=True).mul(100).round(2),
}
)

print()
print(summary)

print()
print(df.info())

print()
print(df.head())


Number of unique users: 90189

         count  percentage
gate_40  45489       50.44
gate_30  44700       49.56

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB
None

   userid  version  sum_gamerounds  retention_1  retention_7
0     116  gate_30               3        False        False
1     337  gate_30              38         True        False
2     377  gate_40             165         True        False
3     483  gate_40               1        False        False
4     488  gate_40             179         True         True


#### 3. Retention rates

Having a game gate at level 40 (treatment) made **fewer** players returning after 7 days compared with the level 30 gate (control). Retention decreased by **-0.82 percentage points** or **-4.3% relative decrease** compared with the control group. However, the observed difference alone does not tell us whether the effect is statistically significant. To test this, I'll use a two-proportion z-test.

In [ ]:
retention = df.groupby("version")["retention_7"].mean()

control_retention = retention.loc["gate_30"]
treatment_retention = retention.loc["gate_40"]

absolute_effect = treatment_retention - control_retention  # Percentage Point Difference
relative_lift = absolute_effect / control_retention  # Proportional Difference

print("Retention by Group:")
print(retention)

print(f"\nControl Retention: {control_retention:.4f}")
print(f"\nTreatment Retention: {treatment_retention:.4f}")
print(f"\nAbsolute Treatment Effect: {absolute_effect:.4f}")
print(f"\nRelative Lift: {relative_lift:.4f}")

Retention by Group:
version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64

Control Retention: 0.1902

Treatment Retention: 0.1820

Absolute Treatment Effect: -0.0082

Relative Lift: -0.0431


### 4. Hypothesis test

I chose a two-sided test because both an increase and a decrease in retention would be relevant to the product decision.

$$
H_0: p_T - p_C = 0
$$

$$
H_1: p_T - p_C \neq 0
$$


With a `p_value` of 0.0016, we can reject the null hypothesis that the retention rates are equal. The results provide evidence that moving the game gate to level 40 reduced 7-day retention.

The treatment reduced 7-day retention by aproximately 0.82 percentage points. The 95% confidence intervall suggests the true treatment effect is likely between -0.31 to -1.33 percentage points

In [76]:
control = df[df["version"]=="gate_30"]
treatment = df[df["version"]=="gate_40"]

# number of successes (retained users) in each group
count = [
    treatment["retention_7"].sum(),
    control["retention_7"].sum()
]

# number of observations (total users) in each group
nobs = [
    treatment["retention_7"].count(),
    control["retention_7"].count()
]

# z-test for two independent proportions
z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

# confidence interval for the difference in proportions
ci = confint_proportions_2indep(
    count1=count[0],
    count2=count[1], 
    nobs1=nobs[0], 
    nobs2=nobs[1], 
    method='wald'
    )

print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"95% Confidence Interval for the Difference in Proportions: {ci[0]:.4f} to {ci[1]:.4f}")


Z-statistic: -3.1644
P-value: 0.0016
95% Confidence Interval for the Difference in Proportions: -0.0133 to -0.0031


### 6. Practical significance

The test results translate to *820 fewer retained user for every 100,000 players* exposed to the game gate at level 40 (treatment).

In a real-world scenario, the product team should ideally decide a certain treshold before the experiment to determine whether the changes are large enough to affect the product decision.

Using a hypotetical treshold of 1% in 7-day retention for demonstration purposes.

While the point estimate of .82 percentage points does not reach 1 percentage point treshold, the 95% confidence interval of -1.33 to -0.31 percentage points includes effects that would exceed the treshold. Therefore the experiment provides evidence of a negative effect, but there is uncertainty about whether its magnitude is meaningful.

In [ ]:
users_per_100k = absolute_effect * 100000
print(f"\nEstimated impact per 100,000 users: {users_per_100k:.0f} fewer retained users")



Estimated impact per 100,000 users: -820 fewer retained users


### 7. Power / Sample Size
